In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, LeaveOneOut, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy.stats import randint

RANDOM_STATE = 42
DATA_PATH = Path("Water Quality.csv")
TARGET_NAME = "Specific conductance"

In [2]:
if not DATA_PATH.exists():
    csvs = list(Path.cwd().glob("*.csv"))
    if len(csvs) == 1:
        DATA_PATH = csvs[0]
        print("Using:", DATA_PATH)
    else:
        raise FileNotFoundError("Put Water Quality.csv beside this notebook or update DATA_PATH.")

raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", raw.shape)
display(raw.head())
required = {"CharacteristicName", "ResultValue"}
if not required.issubset(raw.columns):
    raise ValueError(f"Missing expected columns: {required - set(raw.columns)}")

df = raw.copy()
df["CharacteristicName"] = df["CharacteristicName"].astype(str).str.strip()
df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
keys = [c for c in ["MonitoringLocationID", "ActivityStartDate"] if c in df.columns]
if not keys:
    raise ValueError("Expected MonitoringLocationID and/or ActivityStartDate as sample keys.")
if "ActivityStartDate" in df:
    df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
    df = df.dropna(subset=["ActivityStartDate"])
df = df.dropna(subset=["ResultValue"])
wide = df.pivot_table(index=keys, columns="CharacteristicName", values="ResultValue", aggfunc="mean").reset_index()
wide.columns.name = None
print("Pivoted shape:", wide.shape)
display(wide.head())

Raw shape: (229916, 40)


,DatasetName,MonitoringLocationID,MonitoringLocationName,MonitoringLocationLatitude,MonitoringLocationLongitude,MonitoringLocationHorizontalCoordinateReferenceSystem,MonitoringLocationHorizontalAccuracyMeasure,MonitoringLocationHorizontalAccuracyUnit,MonitoringLocationVerticalMeasure,MonitoringLocationVerticalUnit,...,ResultStatusID,ResultComment,ResultAnalyticalMethodID,ResultAnalyticalMethodContext,ResultAnalyticalMethodName,AnalysisStartDate,AnalysisStartTime,AnalysisStartTimeZone,LaboratoryName,LaboratorySampleID
0,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,110688,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
1,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,2011,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
2,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,20110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
3,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,17209,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN
4,Maritime Coastal Basin Long-term Water Quality...,NS01ED0005,MERSEY RIVER BELOW MILL FALLS SOUTH-WEST OF MA...,44.4367,-65.2228,NAD83,NaN,NaN,NaN,NaN,...,Preliminary,NaN,10110,ENV/CANADA,NaN,NaN,NaN,NaN,ECCCLabs,NaN


Pivoted shape: (7283, 85)


,MonitoringLocationID,ActivityStartDate,"Alkalinity, total",Aluminum,Ammonia,Antimony,Apparent color,Arsenic,Barium,Beryllium,...,True color,Tungsten,Turbidity,Uranium,Vanadium,Ytterbium,Yttrium,Zinc,Zirconium,pH
0,NB01BR0006,2000-09-22,NaN,NaN,NaN,NaN,107.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NB01BR0006,2000-10-11,NaN,NaN,NaN,NaN,274.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NB01BR0006,2000-11-23,NaN,NaN,NaN,NaN,215.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NB01BR0006,2001-04-26,1.8,NaN,NaN,NaN,158.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.9
4,NB01BR0006,2001-05-22,2.8,NaN,NaN,NaN,315.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.1


In [3]:
matches = [c for c in wide.columns if TARGET_NAME.casefold() in str(c).casefold()]
if not matches:
    raise ValueError(f"Target {TARGET_NAME!r} not found. Available measurements: {[c for c in wide.columns if c not in keys][:40]}")
target_col = matches[0]
exclude_terms = ("conductivity", "specific conductance")
features = [c for c in wide.columns if c not in keys and c != target_col
            and not any(t in str(c).casefold() for t in exclude_terms)
            and pd.api.types.is_numeric_dtype(wide[c])]
data = wide[[*keys, target_col, *features]].replace([np.inf, -np.inf], np.nan).dropna(subset=[target_col])
features = [c for c in features if data[c].notna().any() and data[c].nunique(dropna=True) > 1]
if not features:
    raise ValueError("No usable predictor columns after removing target leakage candidates.")
X = data[features].copy()
y = pd.to_numeric(data[target_col], errors="coerce")
valid = y.notna()
X, y, data = X.loc[valid], y.loc[valid], data.loc[valid]

if "ActivityStartDate" in data:
    idx = data["ActivityStartDate"].sort_values().index
    cut = int(0.8 * len(idx))
    train_idx, test_idx = idx[:cut], idx[cut:]
    if not len(train_idx) or not len(test_idx):
        train_idx, test_idx = train_test_split(data.index, test_size=.2, random_state=RANDOM_STATE)
else:
    train_idx, test_idx = train_test_split(data.index, test_size=.2, random_state=RANDOM_STATE)

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Target:", target_col, "| X train:", X_train.shape, "| X test:", X_test.shape)

Target: Specific conductance | X train: (5777, 79) | X test: (1445, 79)


In [4]:
from sklearn.model_selection import KFold
tuning_cv = KFold(n_splits=min(5, len(X_train)), shuffle=True, random_state=RANDOM_STATE)
tune_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1))
])
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": [1.0, "sqrt"]
}
grid_search = GridSearchCV(tune_pipe, param_grid, scoring="neg_mean_absolute_error",
                           cv=tuning_cv, n_jobs=-1, verbose=1, refit=True)
grid_search.fit(X_train, y_train)
print("Grid best params:", grid_search.best_params_)
print("Grid best CV MAE:", -grid_search.best_score_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Grid best params: {'model__max_depth': None, 'model__max_features': 1.0, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Grid best CV MAE: 4.653887028025902


In [5]:
random_distributions = {
    "model__n_estimators": randint(100, 401),
    "model__max_depth": [None, 8, 12, 16, 20, 30],
    "model__min_samples_split": randint(2, 16),
    "model__min_samples_leaf": randint(1, 9),
    "model__max_features": [1.0, "sqrt", 0.7, 0.9]
}
random_search = RandomizedSearchCV(tune_pipe, random_distributions, n_iter=15,
                                   scoring="neg_mean_absolute_error", cv=tuning_cv,
                                   random_state=RANDOM_STATE, n_jobs=-1, verbose=1, refit=True)
random_search.fit(X_train, y_train)
print("Random best params:", random_search.best_params_)
print("Random best CV MAE:", -random_search.best_score_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Random best params: {'model__max_depth': None, 'model__max_features': 1.0, 'model__min_samples_leaf': 3, 'model__min_samples_split': 12, 'model__n_estimators': 269}
Random best CV MAE: 4.844010724083627


In [7]:
comparison = []
for name, estimator in [
                        ("Grid Search best", grid_search.best_estimator_),
                        ("Random Search best", random_search.best_estimator_)]:
    pred = estimator.predict(X_test)
    comparison.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred)**0.5,
        "R2": r2_score(y_test, pred)
    })
tuning_table = pd.DataFrame(comparison).sort_values("MAE")
display(tuning_table.round(4))
tuning_table.to_csv("task6_hyperparameter_tuning_comparison.csv", index=False)

,Model,MAE,RMSE,R2
1,Random Search best,7.5247,37.3208,0.9462
0,Grid Search best,8.0491,38.4855,0.9428
